# Skein Lite: the data architecture, step by step

This notebook walks the data layer that everything else reads: the medallion, the PII
masking, the governed semantic layer, and a governed number. It runs fully offline on the
synthetic domain packs, no keys and no Docker.

Install the extra first: `uv sync --extra notebook`, then run the cells top to bottom.

## 1. Build the medallion

Each domain is a config folder. The manifest drives a bronze -> silver -> gold build in
DuckDB: bronze is the raw CSV as text (lineage), silver is typed and PII-masked, gold is the
curated table the metric layer and the graph read. The same code builds any domain.

In [ ]:
# Bootstrap: make the repo importable and paths stable no matter where the kernel starts
# (nbconvert runs with the cwd set to notebooks/). Run this cell first.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())


In [ ]:
import os, tempfile, duckdb, pandas as pd
import matplotlib.pyplot as plt
from data.lakehouse import build_lakehouse
from data.metrics import MetricResolver

workdir = tempfile.mkdtemp()
apparel_db = os.path.join(workdir, 'apparel.duckdb')
roles = build_lakehouse('apparel_ecommerce', apparel_db)
print('gold tables built:', roles)


## 2. Gold is clean and typed

Gold is what downstream consumers query. Note the types (price is a number, not text).

In [ ]:
con = duckdb.connect(apparel_db, read_only=True)
products = con.execute('select * from main.products').df()
con.close()  # close before opening another connection with a different config
products


## 3. PII never reaches gold raw

The SaaS domain declares `customer_email` as PII. In bronze it is the raw value (lineage);
in silver and gold it is a stable masked pseudonym. A dbt `is_masked` governance test fails
the build if a raw PII value ever lands in gold.

In [ ]:
saas_db = os.path.join(workdir, 'saas.duckdb')
build_lakehouse('saas_support', saas_db)
con = duckdb.connect(saas_db, read_only=True)
raw = con.execute('select customer_email from bronze_tickets limit 3').df()
gold = con.execute('select customer_email from tickets limit 3').df()
con.close()
compare = pd.DataFrame({'bronze (raw, lineage only)': raw['customer_email'],
                        'gold (masked)': gold['customer_email']})
compare


## 4. The semantic layer: one governed number

`metrics.yaml` is the single source of truth. The metric layer runs a validated template on
a read-only connection (never free-form SQL), and the number is cited as evidence. Here is
the return rate by size, straight from gold.

In [ ]:
resolver = MetricResolver('apparel_ecommerce', apparel_db)
result = resolver.resolve('return_rate_by_size', {})  # no params -> all sizes
df = pd.DataFrame(result.rows, columns=result.columns)
df


A quick chart of the same governed numbers:

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(df['size'].astype(str), df['return_rate'], color='#2563eb')
ax.set_ylabel('return rate'); ax.set_xlabel('size')
ax.set_title('Return rate by size (governed)')
plt.tight_layout(); plt.show()


## 5. What this buys us

One tested, documented build. The gold tables carry schema, uniqueness, relationship, and
PII-masking tests on every dbt build; the metric layer sits on top as the single source of
truth; and a parity test proves the in-app builder and dbt produce identical gold. The agent,
the eval, and the dashboards all read the same numbers.

Next: `make dbt-build` runs the tested build, `make dbt-docs` shows the lineage graph, and
`make eval` scores retrieval and the abstain gate.